In [3]:
import pandas as pd

In [4]:
full_charging_df = pd.read_feather('../data/full_charging_df.feather')
full_charging_df.head(5)

,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,...,connectionWeekdayName,connectionYear,isWeekend,connectionHour,disconnectHour,doneChargingHour,loading_duration,connected_duration,ratio_loading_to_connected,invalidDoneDisconnectTimespan
0,5e23b149f9af8b5fe4b973cf,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 09:31:35-08:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,...,Thursday,2020,False,5,11,9,262.683333,362.350000,0.724944,False
1,5e23b149f9af8b5fe4b973d0,2020-01-02 05:36:50-08:00,2020-01-02 14:38:21-08:00,2020-01-02 12:18:05-08:00,33.097,1_1_193_825_2020-01-02 13:36:49.599853,1,AG-1F01,1-1-193-825,America/Los_Angeles,...,Thursday,2020,False,5,14,12,401.250000,541.516667,0.740974,False
2,5e23b149f9af8b5fe4b973d1,2020-01-02 05:56:35-08:00,2020-01-02 16:39:22-08:00,2020-01-02 08:35:06-08:00,6.521,1_1_193_829_2020-01-02 13:56:35.214993,1,AG-1F03,1-1-193-829,America/Los_Angeles,...,Thursday,2020,False,5,16,8,158.516667,642.783333,0.246610,False
3,5e23b149f9af8b5fe4b973d2,2020-01-02 05:59:58-08:00,2020-01-02 08:38:39-08:00,2020-01-02 07:18:45-08:00,2.355,1_1_193_820_2020-01-02 13:59:58.309319,1,AG-1F04,1-1-193-820,America/Los_Angeles,...,Thursday,2020,False,5,8,7,78.783333,158.683333,0.496481,False
4,5e23b149f9af8b5fe4b973d3,2020-01-02 06:00:01-08:00,2020-01-02 14:08:40-08:00,2020-01-02 10:17:30-08:00,13.375,1_1_193_819_2020-01-02 14:00:00.779967,1,AG-1F06,1-1-193-819,America/Los_Angeles,...,Thursday,2020,False,6,14,10,257.483333,488.650000,0.526928,False


In [5]:
# Convert to UTC because i somehow get errors else
full_charging_df['connectionTime_utc'] = full_charging_df['connectionTime'].dt.tz_convert('UTC')
full_charging_df['disconnectTime_utc'] = full_charging_df['disconnectTime'].dt.tz_convert('UTC')

full_charging_df['all_hours'] = full_charging_df.apply(
    lambda row: pd.date_range(
        start=row['connectionTime_utc'].floor('h'),
        end=row['disconnectTime_utc'].ceil('h'),
        freq='h',
        tz='UTC'
    ),
    axis=1
)

hourly_df = full_charging_df.explode('all_hours')
hourly_df['local_hour'] = hourly_df['all_hours'].dt.tz_convert('America/Los_Angeles')

In [6]:
hourly_df[["connectionTime_utc", "disconnectTime_utc", "all_hours",  "doneChargingTime"]].head()

,connectionTime_utc,disconnectTime_utc,all_hours,doneChargingTime
0,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 13:00:00+00:00,2020-01-02 09:31:35-08:00
0,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 14:00:00+00:00,2020-01-02 09:31:35-08:00
0,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 15:00:00+00:00,2020-01-02 09:31:35-08:00
0,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 16:00:00+00:00,2020-01-02 09:31:35-08:00
0,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 17:00:00+00:00,2020-01-02 09:31:35-08:00


## Saturation of used loading stations

In [7]:
hourly_df['local_hour'].head()

0   2020-01-02 05:00:00-08:00
0   2020-01-02 06:00:00-08:00
0   2020-01-02 07:00:00-08:00
0   2020-01-02 08:00:00-08:00
0   2020-01-02 09:00:00-08:00
Name: local_hour, dtype: datetime64[ns, America/Los_Angeles]

## Counting the number of connections per hour

In [8]:
#This dose not include the hours where no one is charging
counting_df = (
    hourly_df
    .groupby(['local_hour'] )
    .agg(session_count=('sessionID', 'nunique'))
    .reset_index()
)

counting_df

,local_hour,session_count
0,2018-04-25 04:00:00-07:00,1
1,2018-04-25 05:00:00-07:00,1
2,2018-04-25 06:00:00-07:00,3
3,2018-04-25 07:00:00-07:00,8
4,2018-04-25 08:00:00-07:00,22
...,...,...
23672,2021-09-14 04:00:00-07:00,1
23673,2021-09-14 05:00:00-07:00,1
23674,2021-09-14 06:00:00-07:00,1
23675,2021-09-14 07:00:00-07:00,1


In [36]:
# Grouped by hour and site
counting_df_by_site = (
    hourly_df
    .groupby(['siteID','local_hour'] )
    .agg(session_count=('sessionID', 'nunique'))
    .reset_index()
)

counting_df_by_site

,siteID,local_hour,session_count
0,1,2018-10-08 06:00:00-07:00,4
1,1,2018-10-08 07:00:00-07:00,19
2,1,2018-10-08 08:00:00-07:00,19
3,1,2018-10-08 09:00:00-07:00,19
4,1,2018-10-08 10:00:00-07:00,19
...,...,...,...
35801,2,2021-09-13 16:00:00-07:00,14
35802,2,2021-09-13 17:00:00-07:00,13
35803,2,2021-09-13 18:00:00-07:00,11
35804,2,2021-09-13 19:00:00-07:00,7


In [37]:
# Add the missing hours
# Generate a range of hours between the min and max time in local_hour
full_range = pd.date_range(start=counting_df_by_site['local_hour'].min(), 
                           end=counting_df_by_site['local_hour'].max(), 
                           freq='h', 
                           tz=counting_df_by_site['local_hour'].dt.tz)

expected_hours = len(pd.date_range(
    start=counting_df_by_site['local_hour'].min(),
    end=counting_df_by_site['local_hour'].max(),
    freq='h',
    tz=counting_df_by_site['local_hour'].dt.tz
))

# Create a cartesian product of siteIDs and full_range
site_ids = counting_df_by_site['siteID'].unique()
full_index = pd.MultiIndex.from_product(
    [site_ids, full_range],
    names=["siteID", "local_hour"]
)

# Create a new DataFrame with all hours and fill missing session_count with 0
full_df = pd.DataFrame(index=full_index).reset_index()

# Merge the original data with the expanded DataFrame
full_df = full_df.merge(counting_df_by_site, on=["siteID", "local_hour"], how="left").fillna({'session_count': 0})

# Convert session_count back to integers if needed
full_df['session_count'] = full_df['session_count'].astype(int)

In [45]:
full_df.head()

,siteID,local_hour,session_count
0,1,2018-04-25 04:00:00-07:00,0
1,1,2018-04-25 05:00:00-07:00,0
2,1,2018-04-25 06:00:00-07:00,0
3,1,2018-04-25 07:00:00-07:00,0
4,1,2018-04-25 08:00:00-07:00,0


In [38]:
# Count rows
expected_hours = len(pd.date_range(
    start=counting_df_by_site['local_hour'].min(),
    end=counting_df_by_site['local_hour'].max(),
    freq='h',
    tz=counting_df_by_site['local_hour'].dt.tz
))
print(f"Expected rows: {expected_hours * 2}")
print(f"Actual rows: {len(full_df)}")

Expected rows: 59434
Actual rows: 59434


In [44]:
# Check timestamp 2018-04-25 04:00:00-07:00
specific_time = pd.Timestamp('2021-09-14 08:00:00-07:00')
counting_df_by_site[counting_df_by_site['local_hour'] == specific_time]

,siteID,local_hour,session_count
15948,1,2021-09-14 08:00:00-07:00,1


In [41]:
# Spot check where session_count is 0
print("Rows where session_count is 0:")
full_df[full_df['session_count'] == 0]

Rows where session_count is 0:


,siteID,local_hour,session_count
0,1,2018-04-25 04:00:00-07:00,0
1,1,2018-04-25 05:00:00-07:00,0
2,1,2018-04-25 06:00:00-07:00,0
3,1,2018-04-25 07:00:00-07:00,0
4,1,2018-04-25 08:00:00-07:00,0
...,...,...,...
59429,2,2021-09-14 04:00:00-07:00,0
59430,2,2021-09-14 05:00:00-07:00,0
59431,2,2021-09-14 06:00:00-07:00,0
59432,2,2021-09-14 07:00:00-07:00,0


## Mean delivered kWh per hour

In [7]:
hourly_df['hour_start'] = hourly_df['local_hour']
hourly_df['hour_end']   = hourly_df['hour_start'] + pd.Timedelta(hours=1)
hourly_df['overlap_start'] = hourly_df[['connectionTime', 'hour_start']].max(axis=1)
hourly_df['overlap_end']   = hourly_df[['disconnectTime', 'hour_end']].min(axis=1)

hourly_df['minutes_in_hour'] = (hourly_df['overlap_end'] - hourly_df['overlap_start']).dt.total_seconds() / 60.0
hourly_df['fraction_of_session'] = hourly_df['minutes_in_hour'] / hourly_df['loading_duration']
hourly_df['kWh_in_this_hour'] = hourly_df['kWhDelivered'] * hourly_df['fraction_of_session']

# Helper to only use valid data
hourly_df = hourly_df[hourly_df['kWh_in_this_hour'] >= 0]

In [8]:
hourly_df['minutes_in_hour']

0        51.100000
0        60.000000
0        60.000000
0        60.000000
0        60.000000
           ...    
65035     3.600000
65036    23.866667
65036    60.000000
65036    60.000000
65036    10.866667
Name: minutes_in_hour, Length: 449980, dtype: float64

In [9]:
full_charging_df[["connectionTime", "disconnectTime", "doneChargingTime", "doneChargingTimespan", "connectionTimespan", "stationID"]].head()

,connectionTime,disconnectTime,doneChargingTime,doneChargingTimespan,connectionTimespan,stationID
0,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 09:31:35-08:00,0 days 04:22:41,0 days 06:02:21,1-1-179-810
1,2020-01-02 05:36:50-08:00,2020-01-02 14:38:21-08:00,2020-01-02 12:18:05-08:00,0 days 06:41:15,0 days 09:01:31,1-1-193-825
2,2020-01-02 05:56:35-08:00,2020-01-02 16:39:22-08:00,2020-01-02 08:35:06-08:00,0 days 02:38:31,0 days 10:42:47,1-1-193-829
3,2020-01-02 05:59:58-08:00,2020-01-02 08:38:39-08:00,2020-01-02 07:18:45-08:00,0 days 01:18:47,0 days 02:38:41,1-1-193-820
4,2020-01-02 06:00:01-08:00,2020-01-02 14:08:40-08:00,2020-01-02 10:17:30-08:00,0 days 04:17:29,0 days 08:08:39,1-1-193-819


In [10]:
def get_is_loading(time: pd.Timestamp, done_charging_time: pd.Timestamp) -> bool:
    if time.hour == done_charging_time.hour:
        return done_charging_time.minute > 29
    return (time + pd.Timedelta(hours=1)) < done_charging_time

In [11]:
hourly_df["is_loading"] = hourly_df.apply(lambda row: get_is_loading(row["local_hour"], row["doneChargingTime"]), axis=1)
hourly_df[["sessionID", "connectionTime", "disconnectTime", "local_hour",  "doneChargingTime", "is_loading"]].head(8)

,sessionID,connectionTime,disconnectTime,local_hour,doneChargingTime,is_loading
0,1_1_179_810_2020-01-02 13:08:53.870034,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 05:00:00-08:00,2020-01-02 09:31:35-08:00,True
0,1_1_179_810_2020-01-02 13:08:53.870034,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 06:00:00-08:00,2020-01-02 09:31:35-08:00,True
0,1_1_179_810_2020-01-02 13:08:53.870034,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 07:00:00-08:00,2020-01-02 09:31:35-08:00,True
0,1_1_179_810_2020-01-02 13:08:53.870034,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 08:00:00-08:00,2020-01-02 09:31:35-08:00,True
0,1_1_179_810_2020-01-02 13:08:53.870034,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 09:00:00-08:00,2020-01-02 09:31:35-08:00,True
0,1_1_179_810_2020-01-02 13:08:53.870034,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 10:00:00-08:00,2020-01-02 09:31:35-08:00,False
0,1_1_179_810_2020-01-02 13:08:53.870034,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 11:00:00-08:00,2020-01-02 09:31:35-08:00,False
1,1_1_193_825_2020-01-02 13:36:49.599853,2020-01-02 05:36:50-08:00,2020-01-02 14:38:21-08:00,2020-01-02 05:00:00-08:00,2020-01-02 12:18:05-08:00,True


In [12]:
kpi_df = hourly_df.groupby('local_hour').agg(
    total_kWh=('kWh_in_this_hour', 'sum'),
    session_count=('sessionID', 'nunique'),
    loading_count=('is_loading', 'mean')
)

kpi_df = kpi_df.reset_index()

In [13]:
kpi_df

,local_hour,total_kWh,session_count,loading_count
0,2018-04-25 04:00:00-07:00,3.094930,1,1.000000
1,2018-04-25 05:00:00-07:00,3.575657,1,1.000000
2,2018-04-25 06:00:00-07:00,3.161296,3,0.666667
3,2018-04-25 07:00:00-07:00,12.883500,7,1.000000
4,2018-04-25 08:00:00-07:00,41.767757,22,0.954545
...,...,...,...,...
23098,2021-09-14 03:00:00-07:00,5.963001,1,1.000000
23099,2021-09-14 04:00:00-07:00,5.963001,1,1.000000
23100,2021-09-14 05:00:00-07:00,5.963001,1,1.000000
23101,2021-09-14 06:00:00-07:00,5.963001,1,1.000000


In [14]:
def get_count(time: pd.Timestamp):
    try:
        return kpi_df.loc[kpi_df["local_hour"] == time]["session_count"].iloc[0]
    except:
        return 0

In [15]:
specific_time = pd.Timestamp('2019-08-18 22:00:00-08:00')
get_count(specific_time)

np.int64(1)

In [10]:
# kpi_df.to_feather('../data/kpi_df.feather')
counting_df_by_site.to_feather('../data/counting_df_by_site.feather')